# 🧠 Pelatihan Model AI TexCycle (MobileNetV2 TFLite)
### Deteksi & Klasifikasi 8 Jenis Limbah Tekstil (B3 & Non-B3)
---
Notebook ini digunakan untuk melatih model klasifikasi citra limbah tekstil untuk aplikasi mobile **TexCycle**.

**8 Kelas Limbah Tekstil:**
1. `kain_besar` (Non-B3 - Potongan kain perca ukuran besar > 30 cm)
2. `kain_sedang` (Non-B3 - Potongan kain perca ukuran 10-30 cm)
3. `kain_kecil` (Non-B3 - Sisa perca kecil < 10 cm)
4. `benang` (Non-B3 - Sisa benang jahit/kelos)
5. `kemasan` (Non-B3 - Kemasan plastik/karton gulungan kain)
6. `limbah_cair` (B3 - Cairan pewarna/kimia sisa proses tekstil)
7. `sludge` (B3 - Lumpur endapan IPAL tekstil)
8. `majun` (B3 - Kain majun kotor pelumas/oli/bahan kimia)

**Target Output:**
- `texcycle_model.tflite` (Model terkuantisasi < 5 MB, siap jalan offline di HP)
- `labels.txt` (Daftar nama kelas)
- Grafik Evaluasi & Confusion Matrix (Bahan dokumentasi Paper)

## 1. Import Library & Periksa Akselerasi GPU

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
from PIL import Image

print("TensorFlow Version:", tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print("GPU Aktif:", gpus[0])
else:
    print("PERINGATAN: Berjalan di CPU. Di Colab, pilih Runtime > Change runtime type > T4 GPU untuk proses kilat.")

## 2. Struktur Direktori Dataset & Generator Starter Data
Jika Anda belum mengunggah foto asli dari lapangan, sel berikut akan otomatis membuat struktur folder dan men-generate sampel data awal agar pipeline training dapat diuji langsung dari awal sampai akhir.

In [ ]:
CLASSES = [
    'kain_besar',
    'kain_sedang',
    'kain_kecil',
    'benang',
    'kemasan',
    'limbah_cair',
    'sludge',
    'majun'
]

DATASET_DIR = 'dataset_texcycle'
os.makedirs(DATASET_DIR, exist_ok=True)

# Buat folder untuk tiap kelas
for c in CLASSES:
    os.makedirs(os.path.join(DATASET_DIR, c), exist_ok=True)

# Periksa apakah sudah ada data di folder
total_images = sum([len(files) for r, d, files in os.walk(DATASET_DIR)])

if total_images < 40:
    print("Menghasilkan sampel starter data untuk 8 kelas...")
    colors = [
        (70, 130, 180),  # kain_besar (steel blue)
        (100, 149, 237), # kain_sedang (cornflower blue)
        (135, 206, 250), # kain_kecil (light sky blue)
        (255, 165, 0),   # benang (orange)
        (218, 165, 32),  # kemasan (goldenrod)
        (34, 139, 34),   # limbah_cair (forest green / zat kimia)
        (47, 79, 79),    # sludge (dark slate gray / lumpur)
        (105, 105, 105)  # majun (dim gray / kotor)
    ]
    
    for i, c in enumerate(CLASSES):
        class_dir = os.path.join(DATASET_DIR, c)
        base_col = colors[i]
        for j in range(25): # 25 gambar starter per kelas
            noise = np.random.randint(-35, 35, (224, 224, 3))
            img_arr = np.clip(np.ones((224, 224, 3)) * base_col + noise, 0, 255).astype(np.uint8)
            img = Image.fromarray(img_arr)
            img.save(os.path.join(class_dir, f"{c}_{j+1:03d}.jpg"))
            
    print("Starter dataset berhasil dibuat!")
else:
    print(f"Ditemukan {total_images} gambar dalam direktori dataset.")

## 3. Data Loading & Augmentasi Citra
Resolusi input disesuaikan standar mobile vision ($224 \times 224$ piksel) dengan pemisahan $80\%$ Data Latih dan $20\%$ Data Validasi.

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 16

train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.15),
    layers.RandomContrast(0.2)
])

## 4. Arsitektur Model: MobileNetV2 (Transfer Learning)
MobileNetV2 dirancang khusus oleh Google untuk inferensi berkecepatan tinggi pada smartphone dengan konsumsi daya minimal.

In [ ]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False  # Freeze pretrained weights ImageNet

inputs = tf.keras.Input(shape=(224, 224, 3))
x = data_augmentation(inputs)
x = layers.Rescaling(1./127.5, offset=-1)(x)  # Normalisasi ke rentang [-1, 1]
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.3)(x)
x = layers.Dense(256, activation='relu')(x)
outputs = layers.Dense(len(CLASSES), activation='softmax')(x)

model = tf.keras.Model(inputs, outputs, name="TexCycle_Classifier")

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

## 5. Proses Training Model

In [ ]:
EPOCHS = 15
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True)
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks
)

## 6. Visualisasi Evaluasi Training (Bahan Gambar untuk Paper)
Sel ini menyimpan grafik akurasi dan loss ke file PNG (`evaluasi_training.png`) yang bisa dicantumkan di Paper.

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy', color='#2196F3', linewidth=2)
plt.plot(history.history['val_accuracy'], label='Val Accuracy', color='#4CAF50', linewidth=2)
plt.title('Kurva Akurasi Model TexCycle')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss', color='#F44336', linewidth=2)
plt.plot(history.history['val_loss'], label='Val Loss', color='#FF9800', linewidth=2)
plt.title('Kurva Loss Model TexCycle')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('evaluasi_training.png', dpi=300)
plt.show()
print("Grafik disimpan sebagai: evaluasi_training.png")

## 7. Evaluasi Confusion Matrix (Bahan Tabel Paper)

In [ ]:
all_preds = []
all_true = []

for images, labels in val_ds:
    preds = model.predict(images, verbose=0)
    all_preds.extend(np.argmax(preds, axis=1))
    all_true.extend(np.argmax(labels.numpy(), axis=1))

print("--- CLASSIFICATION REPORT ---")
print(classification_report(all_true, all_preds, target_names=CLASSES, zero_division=0))

cm = confusion_matrix(all_true, all_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CLASSES, yticklabels=CLASSES)
plt.title('Confusion Matrix Klasifikasi Limbah Tekstil')
plt.xlabel('Prediksi Model')
plt.ylabel('Label Aktual')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=300)
plt.show()
print("Confusion matrix disimpan sebagai: confusion_matrix.png")

## 8. Konversi ke TensorFlow Lite (`.tflite`) & Kuantisasi
Melakukan kompresi model agar ukuran berkas sangat kecil (sekitar 3-5 MB) dan waktu inferensi di smartphone menjadi instan.

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_quantized_model = converter.convert()

tflite_model_path = 'texcycle_model.tflite'
with open(tflite_model_path, 'wb') as f:
    f.write(tflite_quantized_model)

size_mb = os.path.getsize(tflite_model_path) / (1024 * 1024)
print(f"✅ Model TFLite berhasil dibuat: {tflite_model_path}")
print(f"📦 Ukuran Model: {size_mb:.2f} MB (Sangat ringan untuk HP)")

# Simpan labels.txt
labels_path = 'labels.txt'
with open(labels_path, 'w') as f:
    for c in CLASSES:
        f.write(c + '\n')
print(f"✅ File label berhasil disimpan: {labels_path}")

## 9. Unduh Hasil Model untuk Dimasukkan ke Flutter

In [ ]:
try:
    from google.colab import files
    files.download('texcycle_model.tflite')
    files.download('labels.txt')
    files.download('evaluasi_training.png')
    files.download('confusion_matrix.png')
    print("Pengunduhan otomatis dijalankan.")
except ImportError:
    print("Berjalan di lokal. File telah disimpan di direktori kerja aktif.")